[◀ Appendix B: Regex](Appendix_B_regular_expressions.ipynb) | [Table of Contents](TOC.ipynb)

# Appendix C: Serialization & Serialization Formats

This appendix covers serialization formats in Python—specifically **JSON**, **CSV**, and **Pickle**—along with file streaming, custom encoder behaviors, and crucial security warnings.

Reference: [Data Persistence](https://docs.python.org/3/library/persistence.html)


## 1. JSON (JavaScript Object Notation)

JSON is a lightweight, language-agnostic text serialization format. Python's built-in `json` module provides tools to serialize (`dumps`/`dump`) and deserialize (`loads`/`load`) standard data structures.

### Custom JSON Encoding
JSON natively supports only a few basic types: `dict`, `list`, `str`, `int`, `float`, `bool`, and `None`. Trying to serialize complex objects (like `datetime` instances or custom classes) raises a `TypeError`.

To serialize custom objects, you can:
1. Supply a custom function to the `default` parameter of `json.dumps()`.
2. Inherit from `json.JSONEncoder` and override the `default` method.

### Custom JSON Decoding using Object Hooks
While serialization converts Python objects to JSON text, deserialization (`json.loads`/`json.load`) reconstructs JSON text back into standard Python structures. By default, JSON objects `{}` are deserialized into Python `dict` instances.

To reconstruct custom class objects (or convert strings back to `datetime` objects) during deserialization, Python provides the **`object_hook`** parameter:
- `object_hook` accepts a callable that is invoked with every deserialized dictionary (`dict`) object.
- The hook function inspects the dictionary keys and values. If they match a specific signature (e.g. containing metadata markers like `__task__` or `__class__`), it creates and returns an instance of the custom class. Otherwise, it returns the dictionary as-is.

#### CPython Decoder Execution Flow
Under the hood, CPython's `json.loads()` uses the `json.decoder.JSONDecoder` class. When the parser encounters a JSON object (`{}`), it parses it into a standard Python `dict`. It then immediately passes this dictionary to the `object_hook` function. The substitution happens **bottom-up (post-order traversal)**; inner dictionaries are passed to `object_hook` and replaced before outer dictionaries containing them are processed.

#### Reconstructing ISO Dates: `datetime.fromisoformat()`
To serialize a `datetime` object, it is typically converted to an ISO 8601 string (e.g. `2026-06-02T12:00:00`). To reconstruct this back to a `datetime` object during deserialization, use the built-in class method **`datetime.fromisoformat(date_string)`** (introduced in Python 3.7).

#### Detailed Code Example
```python
import json
from datetime import datetime

class Task:
    def __init__(self, title, deadline, completed):
        self.title = title
        self.deadline = deadline
        self.completed = completed

def task_decoder_hook(dct):
    # Check for metadata marker
    if "__task__" in dct:
        # Reconstruct datetime from ISO string
        deadline = datetime.fromisoformat(dct["deadline"])
        return Task(dct["title"], deadline, dct["completed"])
    return dct

json_data = '''
[
  {
    "__task__": true,
    "title": "Review PEP 8",
    "deadline": "2026-06-02T14:00:00",
    "completed": false
  }
]
'''

tasks = json.loads(json_data, object_hook=task_decoder_hook)
print(tasks)        # List of [Task object]
print(type(tasks[0].deadline))  # <class 'datetime.datetime'>
```


In [ ]:
import json
from datetime import datetime

class Event:
    def __init__(self, name, date):
        self.name = name
        self.date = date

    def __repr__(self):
        return f"Event(name={self.name!r}, date={self.date})"

# Custom Encoder
class CustomEventEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, datetime):
            return obj.isoformat()  # Convert datetime to ISO string
        if isinstance(obj, Event):
            # Include a metadata marker for identification during decoding
            return {"__event__": True, "name": obj.name, "timestamp": obj.date}
        return super().default(obj)

# Custom Decoder Hook
def event_decoder_hook(dct):
    if "__event__" in dct:
        # Reconstruct the datetime from the ISO 8601 string
        date_val = datetime.fromisoformat(dct["timestamp"])
        return Event(dct["name"], date_val)
    return dct

event = Event("System Launch", datetime(2026, 6, 2, 12, 0, 0))

# 1. Serialize using custom encoder
json_str = json.dumps(event, cls=CustomEventEncoder, indent=2)
print("Serialized JSON String:")
print(json_str)

# 2. Deserialize using object_hook and fromisoformat()
reconstructed_event = json.loads(json_str, object_hook=event_decoder_hook)
print("\nDeserialized Reconstructed Event:")
print(reconstructed_event)
print("Reconstructed date type:", type(reconstructed_event.date))


## 2. CSV (Comma-Separated Values)

The `csv` module is optimized for tabular records. Rather than parsing CSV lines manually (which breaks on embedded commas or quoting rules), use Python's built-in classes:
- **`csv.DictWriter`** & **`csv.DictReader`**: Map columns to dictionaries, making them much more readable and robust compared to coordinate-based list indexes.


In [ ]:
import csv
import io

# We use StringIO to simulate writing to a file in memory
csv_buffer = io.StringIO()
headers = ["userid", "role", "status"]

# Writing CSV with DictWriter
writer = csv.DictWriter(csv_buffer, fieldnames=headers)
writer.writeheader()
writer.writerow({"userid": 101, "role": "admin", "status": "active"})
writer.writerow({"userid": 102, "role": "user", "status": "pending"})

print("CSV Content:\n", csv_buffer.getvalue())

# Reading CSV with DictReader
csv_buffer.seek(0)
reader = csv.DictReader(csv_buffer)
for row in reader:
    print(f"Row: ID={row['userid']}, Role={row['role']}")


## 3. Pickle: Python Object Persistence

`pickle` is a Python-specific binary serialization protocol. It can serialize almost any Python object (including closures, functions, and custom instances) without writing manual serializers.

### ⚠️ CRITICAL SECURITY WARNING: Pickle Exploit
**Never load pickle files from untrusted sources.** 
The pickle protocol works by executing virtual machine instructions. A class can define a custom `__reduce__` method that returns a callable and arguments to execute upon deserialization. If a user loads a malicious pickle file, **it can execute arbitrary shell commands or code on the server**.


In [ ]:
import pickle
import subprocess

# 1. Basic usage
data = {'key': 'value', 'numbers': [1, 2, 3]}
serialized = pickle.dumps(data)
print("Serialized size:", len(serialized), "bytes")

# 2. Malicious Exploit Demonstration
class Exploit:
    def __reduce__(self):
        # This callable will run automatically when loaded
        # We simulate running print('SYSTEM COMPROMISED!')
        return (print, ("⚠️ SYSTEM COMPROMISED! Malicious code executed via pickle!",))

malicious_payload = pickle.dumps(Exploit())

print("\nLoading untrusted pickle payload:")
pickle.loads(malicious_payload)


## 4. Coding Challenges

Complete the following exercises to test your understanding.


### Challenge 1: Configuration Data Exporter

Implement the function `export_config(json_str, csv_str)` which consolidates settings from two input formats:
1. **`json_str`**: A JSON string containing base configurations (e.g. `{"host": "localhost", "port": 8080}`).
   - Parse the JSON configuration. If a `json.JSONDecodeError` occurs, catch it and raise a custom `ConfigParseError` with the message `"Invalid JSON configuration"` **chained** from the original `JSONDecodeError`.
2. **`csv_str`**: A CSV string containing environmental override options. The CSV always contains headers `key` and `value` (e.g., `key,value\nport,9090\n` etc).
   - Parse the CSV. If an error occurs, catch it and raise `ConfigParseError` with the message `"Invalid CSV configuration"` chained from the original exception.
   - If a value in the CSV consists only of digits, convert it to an **integer** type. If it is exactly `'true'` or `'false'` (case-insensitive), convert it to the corresponding **boolean** value.
3. **Merge**: Any keys found in the CSV overrides should update or add keys to the base JSON settings.
4. Return the consolidated configuration dictionary.


In [ ]:
import json
import csv
import io

class ConfigError(Exception):
    pass

class ConfigParseError(ConfigError):
    pass

def export_config(json_str: str, csv_str: str) -> dict:
    """
    Reads JSON base settings and CSV overrides, merging them into a final dictionary.
    Converts CSV integers and boolean values accordingly.
    Chains parsing errors into ConfigParseError.
    """
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Run this cell to verify your implementation
try:
    # Test case 1: Successful merge and conversion
    json_input = '{"host": "localhost", "port": 8080, "debug": false}'
    csv_input = "key,value\nport,9000\ndebug,true\napi_key,xyz123\n"
    
    config = export_config(json_input, csv_input)
    assert config['host'] == "localhost", "Should retain unchanged JSON keys"
    assert config['port'] == 9000, f"Should convert port override to int, got: {type(config['port'])}"
    assert config['debug'] is True, "Should convert debug override to True boolean"
    assert config['api_key'] == "xyz123", "Should add new keys from CSV"
    
    # Test case 2: JSON decode failure exception chaining
    try:
        export_config("{bad json", "")
        raise AssertionError("Failed to raise ConfigParseError on malformed JSON")
    except ConfigParseError as e:
        assert isinstance(e.__cause__, json.JSONDecodeError), "Original JSON exception was not chained"
        assert str(e) == "Invalid JSON configuration", f"Incorrect exception message: {e}"
        
    print("🎉 Challenge 1 Passed Successfully!")
except AssertionError as e:
    print(f"❌ Test Failed: {e}")


### Challenge 2: Custom Class Serialization with Datetime Support

Implement a task management serialization system using JSON.
We define a custom class `Task`:
```python
class Task:
    def __init__(self, title: str, deadline: datetime, completed: bool):
        self.title = title
        self.deadline = deadline
        self.completed = completed
```

You must implement:
1. **`TaskEncoder`**: A custom `json.JSONEncoder` subclass that encodes `Task` instances into a JSON-compatible dictionary. Mark the dictionary with a special key `"__task__"` set to `True` so the decoder knows it represents a `Task`. Convert the `deadline` (a `datetime` object) into an ISO-formatted string using `.isoformat()`.
2. **`load_tasks(json_str)`**: A function that parses a JSON string representing a list of tasks and reconstructs the original Python list of `Task` objects, reconstructing the `deadline` back into a proper `datetime` object.


In [ ]:
class Task:
    def __init__(self, title: str, deadline: datetime, completed: bool):
        self.title = title
        self.deadline = deadline
        self.completed = completed

    def __eq__(self, other):
        return (
            isinstance(other, Task) and
            self.title == other.title and
            self.deadline == other.deadline and
            self.completed == other.completed
        )

class TaskEncoder(json.JSONEncoder):
    # Overload default here
    # YOUR CODE HERE
    pass

def load_tasks(json_str: str) -> list[Task]:
    # Implement load logic and object_hook
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Run this cell to verify your implementation
try:
    t1 = Task("Review PEP 8", datetime(2026, 6, 2, 14, 0), False)
    t2 = Task("Release Python Beta", datetime(2026, 8, 1, 9, 30), True)
    tasks = [t1, t2]
    
    # Serialize
    serialized_tasks = json.dumps(tasks, cls=TaskEncoder, indent=2)
    
    # Verify serialization keys
    raw_parsed = json.loads(serialized_tasks)
    assert len(raw_parsed) == 2
    assert raw_parsed[0]['__task__'] is True, "Missing task metadata marker"
    assert isinstance(raw_parsed[0]['deadline'], str), "Deadline should be serialized as an ISO string"
    
    # Deserialize
    deserialized_tasks = load_tasks(serialized_tasks)
    assert len(deserialized_tasks) == 2, "Should deserialize 2 tasks"
    assert deserialized_tasks[0] == t1, "First task mismatch"
    assert deserialized_tasks[1] == t2, "Second task mismatch"
    assert isinstance(deserialized_tasks[0].deadline, datetime), "Deadline should be a datetime object"
    
    print("🎉 Challenge 2 Passed Successfully!")
except AssertionError as e:
    print(f"❌ Test Failed: {e}")


[◀ Appendix B: Regex](Appendix_B_regular_expressions.ipynb) | [Table of Contents](TOC.ipynb)